# Notebook 32 — Rebuild Adaptive Test and Lock the Final Test Protocol

Notebook 31 showed a promising Corrected V4 + author-affinity point estimate, but its confidence interval versus Legacy V4.2 still crossed zero. This notebook prepares an independent test set before any further model decision.

It performs data work only:

1. rebuild the previously incomplete `sequence_examples/test` from complete Two-Tower Gold;
2. enforce strict-prior sequence timestamps and validate full test-key coverage;
3. create a deterministic exact-target A2 test protocol;
4. upload the large artifacts directly to private Hugging Face storage;
5. save only compact manifests/logs to Drive and your computer.

Use a **CPU High-RAM** Colab runtime. A GPU provides no benefit for this notebook. Notebook 33 will perform corrected-sequence reconstruction and the final GPU evaluation separately.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

from getpass import getpass
from pathlib import Path
import hashlib, importlib.util, json, os, shutil, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
BASE_GOLD_ROOT = DRIVE_ROOT / 'data/gold/two_tower/v1_colab'
DRIVE_ADAPTIVE_ROOT = DRIVE_ROOT / 'data/gold/adaptive/v1'
SEQUENCE_CONTEXT = DRIVE_ADAPTIVE_ROOT / '_intermediate/sequence_context'
WORK_ROOT = Path('/content/recsys_adaptive_test_v1')
SOURCE_ROOT = WORK_ROOT / 'runtime'
OUTPUT_ROOT = WORK_ROOT / 'adaptive_test_v1'
PROTOCOL_DIR = OUTPUT_ROOT / 'evaluation_protocol/v2_test'
DRIVE_REPORT_DIR = DRIVE_ROOT / 'artifacts/adaptive/test_rebuild_v1'
SPARK_LOCAL_DIR = Path('/content/spark-local')
for path in (SOURCE_ROOT, OUTPUT_ROOT, PROTOCOL_DIR, DRIVE_REPORT_DIR, SPARK_LOCAL_DIR):
    path.mkdir(parents=True, exist_ok=True)
print('work root:', WORK_ROOT)

In [ ]:
required = {
    'pyspark': 'pyspark==3.5.6', 'pandas': 'pandas', 'pyarrow': 'pyarrow',
    'huggingface_hub': 'huggingface_hub>=0.32',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
free_gib = shutil.disk_usage('/content').free / 1024**3
ram_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3
assert free_gib >= 25, f'At least 25 GiB local disk is required; only {free_gib:.1f} GiB is free.'
assert ram_gib >= 20, f'Use a CPU High-RAM runtime; only {ram_gib:.1f} GiB RAM is available.'
print(f'RAM: {ram_gib:.1f} GiB | local disk free: {free_gib:.1f} GiB')

In [ ]:
required_paths = [
    BASE_GOLD_ROOT / 'targets/test',
    BASE_GOLD_ROOT / 'user_state/test',
    BASE_GOLD_ROOT / 'item_features/point_in_time/test',
    DRIVE_ADAPTIVE_ROOT / 'sequence_examples/train',
    SEQUENCE_CONTEXT / '_SUCCESS',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, 'Missing Drive inputs:\n' + '\n'.join(missing_paths)
def parquet_count(path):
    return len([p for p in Path(path).rglob('*.parquet') if not p.name.startswith('.')])
for path in required_paths[:-1]:
    print(path, 'parquet files:', parquet_count(path))
assert parquet_count(BASE_GOLD_ROOT / 'targets/test') > 0
assert parquet_count(SEQUENCE_CONTEXT) > 0
print('Drive preflight: PASS')

## Verified runtime and Hugging Face write preflight

Enable notebook access for `HF_TOKEN` and `HF_WRITE_TOKEN` in Colab Secrets. The write probe runs before the expensive Spark build.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

HF_REPO_ID = 'khoa1505/recsys-artifacts'
RUNTIME_FILE = 'runtime/adaptive_test_rebuild_runtime_v1.zip'
RUNTIME_SHA256 = 'e8e3adda1fb4cd0e4b1b441a8686d062d76c490b2bce824da5a4e2933707fcec'

def normalize_token(raw):
    if isinstance(raw, bytes): raw = raw.decode()
    if isinstance(raw, str):
        raw = raw.strip()
        return raw if raw.startswith('hf_') else None
    if isinstance(raw, dict):
        for value in raw.values():
            token = normalize_token(value)
            if token: return token
    return None

def secret(name):
    try: return normalize_token(userdata.get(name))
    except Exception: return None

HF_TOKEN = secret('HF_TOKEN') or normalize_token(getpass('Paste Hugging Face READ token: '))
HF_WRITE_TOKEN = secret('HF_WRITE_TOKEN') or normalize_token(getpass('Paste Hugging Face WRITE token: '))
assert HF_TOKEN and HF_WRITE_TOKEN, 'Both Hugging Face tokens are required.'
api = HfApi(token=HF_WRITE_TOKEN)
probe = WORK_ROOT / 'write_probe.txt'; probe.write_text('adaptive-test-write-probe\n')
api.upload_file(repo_id=HF_REPO_ID, repo_type='model', path_or_fileobj=str(probe), path_in_repo='adaptive_test/.write_probe.txt', commit_message='Verify adaptive test write access')
api.delete_file(repo_id=HF_REPO_ID, repo_type='model', path_in_repo='adaptive_test/.write_probe.txt', commit_message='Remove adaptive test write probe')

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        while block := stream.read(8 * 1024 * 1024): digest.update(block)
    return digest.hexdigest()

runtime_zip = Path(hf_hub_download(HF_REPO_ID, RUNTIME_FILE, repo_type='model', token=HF_TOKEN))
assert sha256(runtime_zip) == RUNTIME_SHA256, 'Runtime ZIP hash mismatch'
print('Hugging Face/runtime preflight: PASS')

In [ ]:
if SOURCE_ROOT.exists(): shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    assert archive.testzip() is None
    archive.extractall(SOURCE_ROOT)
BUILD_SCRIPT = SOURCE_ROOT / 'scripts/build_adaptive_test_artifacts.py'
PROTOCOL_SCRIPT = SOURCE_ROOT / 'scripts/build_adaptive_eval_protocol_v2.py'
assert BUILD_SCRIPT.exists() and PROTOCOL_SCRIPT.exists()
compile(BUILD_SCRIPT.read_text(), str(BUILD_SCRIPT), 'exec')
compile(PROTOCOL_SCRIPT.read_text(), str(PROTOCOL_SCRIPT), 'exec')
print('runtime extraction/compile: PASS')

## Rebuild the adaptive test split

The output is written to Colab local disk, not Drive. Spark validates full key coverage, uniqueness, user/item joins and strict-prior timestamps before the next cell can run.

In [ ]:
shutil.rmtree(OUTPUT_ROOT / 'sequence_examples/test', ignore_errors=True)
env = os.environ.copy()
env['SPARK_MASTER'] = 'local[2]'
env['SPARK_LOCAL_DIR'] = str(SPARK_LOCAL_DIR)
env['PYSPARK_SUBMIT_ARGS'] = '--driver-memory 14g pyspark-shell'
command = [
    sys.executable, str(BUILD_SCRIPT),
    '--base-gold-root', str(BASE_GOLD_ROOT),
    '--sequence-context', str(SEQUENCE_CONTEXT),
    '--output-dir', str(OUTPUT_ROOT),
    '--shuffle-partitions', '96', '--output-partitions', '96', '--overwrite',
]
build_log = Path('/content/adaptive_test_rebuild.log')
process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with build_log.open('w') as log:
    for line in process.stdout:
        print(line, end=''); log.write(line); log.flush()
code = process.wait()
if code != 0:
    tail = '\n'.join(build_log.read_text(errors='replace').splitlines()[-240:])
    raise RuntimeError(f'Adaptive test rebuild failed with exit code {code}:\n{tail}')
test_manifest = json.loads((OUTPUT_ROOT / 'manifest.json').read_text())
assert test_manifest['quality']['passed'] is True
print('adaptive test quality: PASS')

## Build the locked A2 test protocol

The protocol builder reads the existing Drive train split directly only to derive train-warm item labels. Training data is never copied into the new artifact.

In [ ]:
shutil.rmtree(PROTOCOL_DIR, ignore_errors=True)
protocol_log = Path('/content/adaptive_test_protocol.log')
command = [
    sys.executable, str(PROTOCOL_SCRIPT), '--adaptive-root', str(OUTPUT_ROOT),
    '--output-dir', str(PROTOCOL_DIR), '--split', 'test',
    '--train-source', str(DRIVE_ADAPTIVE_ROOT / 'sequence_examples/train'),
    '--max-queries-per-user', '12', '--seed', '44', '--overwrite',
]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with protocol_log.open('w') as log:
    for line in process.stdout:
        print(line, end=''); log.write(line); log.flush()
code = process.wait()
if code != 0:
    tail = '\n'.join(protocol_log.read_text(errors='replace').splitlines()[-240:])
    raise RuntimeError(f'Test protocol build failed with exit code {code}:\n{tail}')
protocol_manifest = json.loads((PROTOCOL_DIR / 'manifest.json').read_text())
assert protocol_manifest['split'] == 'test'
assert protocol_manifest['builder_revision'] == '2.1-session-history'
assert protocol_manifest['queries'] > 0 and protocol_manifest['users'] > 0
print('locked test protocol: PASS')
print('queries/users:', protocol_manifest['queries'], protocol_manifest['users'])

## Upload once to private Hugging Face storage

Only the rebuilt test split and compact protocol are uploaded. The existing train split is not part of this folder.

In [ ]:
assert not (OUTPUT_ROOT / 'sequence_examples/train').exists()
api.upload_folder(
    repo_id=HF_REPO_ID, repo_type='model', folder_path=str(OUTPUT_ROOT),
    path_in_repo='adaptive_test/v1', commit_message='Upload rebuilt adaptive test and locked protocol',
)
assert api.file_exists(HF_REPO_ID, 'adaptive_test/v1/manifest.json', repo_type='model')
assert api.file_exists(HF_REPO_ID, 'adaptive_test/v1/evaluation_protocol/v2_test/queries.parquet', repo_type='model')
remote_files = [name for name in api.list_repo_files(HF_REPO_ID, repo_type='model') if name.startswith('adaptive_test/v1/sequence_examples/test/') and name.endswith('.parquet')]
assert len(remote_files) > 0, 'No remote test Parquet files found after upload'
print('Hugging Face artifact upload: PASS')
print('remote test parquet files:', len(remote_files))

In [ ]:
report_dir = WORK_ROOT / 'small_report'
shutil.rmtree(report_dir, ignore_errors=True); report_dir.mkdir()
shutil.copy2(OUTPUT_ROOT / 'manifest.json', report_dir / 'adaptive_test_manifest.json')
shutil.copy2(PROTOCOL_DIR / 'manifest.json', report_dir / 'test_protocol_manifest.json')
shutil.copy2(PROTOCOL_DIR / 'queries_preview.csv', report_dir / 'test_queries_preview.csv')
shutil.copy2(build_log, report_dir / build_log.name)
shutil.copy2(protocol_log, report_dir / protocol_log.name)
location = {
    'hf_repo': HF_REPO_ID, 'hf_prefix': 'adaptive_test/v1',
    'test_parquet_files': len(remote_files),
    'queries': protocol_manifest['queries'], 'users': protocol_manifest['users'],
}
(report_dir / 'artifact_location.json').write_text(json.dumps(location, indent=2))
for path in report_dir.iterdir(): shutil.copy2(path, DRIVE_REPORT_DIR / path.name)
archive = Path(shutil.make_archive('/content/adaptive_test_v1_report', 'zip', report_dir))
print('Drive report:', DRIVE_REPORT_DIR)
print('download:', archive, f'{archive.stat().st_size / 1024**2:.2f} MiB')
files.download(str(archive))

## Next step

Send back `adaptive_test_v1_report.zip`. Notebook 33 will then reconstruct corrected strict-prior sequences only for the locked test queries and evaluate the frozen Corrected V4 + author policy against Legacy V4.2. No weight will be tuned on test.